# Libraries

In [1]:
# Standard Libraries
import os
from natsort import natsorted
from matplotlib import pyplot as plt

# EEG & Signal Processing
import mne
import asrpy
from mne.preprocessing import ICA

# Data Manipulation
import numpy as np
import pandas as pd

# PyTorch Libraries
import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torch import optim
from tqdm import tqdm

import torchvision

import torch.nn.functional as F
import torchvision.transforms as transforms

from torchmetrics import MetricCollection
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassRecall

# Training split
from sklearn.model_selection import train_test_split

# Feature scaling
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

# Function to find .set files

In [3]:
def find_files(data_path, file_extension):

    all_files = []  # stores loaded EEG data

    for root, _, files in os.walk(data_path):
        for file in files:
            if file.endswith(file_extension):
                full_file_path = os.path.join(
                    root, file
                )  # creates full file path separately for each .set file
                all_files.append(full_file_path)

    return all_files

# Preprocessing Function
Brain data must be cleaned before data analysis. The following function uses a Butterworth band-pass filter which ranges from 0.5 - 45 Hz, rereferencing the the average of all the brain data, artificat subspace reconstruction, and independent component analysis which gets rid of noise due to possible eye or muscle movements. 

In [4]:
def preprocess(raw):

    # raw = raw.copy()
    raw.load_data()

    # Butterworth band-pass filter (0.5-45 Hz)
    raw.filter(
        l_freq=0.5,
        h_freq=45,
        method="iir",
        iir_params=dict(order=4, ftype="butter"),
    )
    # Rereference to CAR (Common Average Reference)
    raw.set_eeg_reference(
        ref_channels="average", projection=False
    )  # copy = True by default meaning this doesn't change OG data files

    # ASR Routine with SD = 17
    asr = asrpy.ASR(
        sfreq=float(raw.info["sfreq"]),
        cutoff=17,
        win_len=0.5,
    )
    asr.fit(raw, picks="eeg")
    asr.transform(raw, picks="eeg")

    # ICA to remove ocular artifacts (eye blinks, muscle movements...)
    ica = ICA(
        n_components=0.99,
        method="fastica",
        random_state=99, 
        max_iter="auto",
        # fit_params=dict(extended=True), Q: Do I need this line? and picks="eeg" ?
    )
    # picks="eeg",
    ica.fit(raw, verbose=False)
    ica.apply(raw)

    return raw

# Feature Engineering Functions

In [5]:
# Function to normalize features using min-max scaling
def normalize (features):
    min_val = np.min(features, axis = (1,2,3))
    max_val = np.max(features, axis = (1,2,3))
    normalized_features = (features - min_val) / (max_val - min_val)
    return normalized_features

# Function to standardize features using z-score calculation
def standardize(features):
    mean = np.mean(features, axis = (1,2,3))
    std = np.std(features, axis = (1,2,3))
    standardized_features = (features - mean) / std
    return standardized_features

def robustScale(features):
    median = np.median(features, axis = (1,2,3))
    q1 = np.percentile(features, 25, axis = (1,2,3))
    q3 = np.percentile(features, 75, axis = (1,2,3))
    iqr = q3 - q1
    scaled_features = (features - median) / iqr
    return scaled_features

# Per batch
def batch_normalization(batch):
    for batch in train_dataloader:
        features, labels = batch
        print("Original feature shape: ", features.shape)  # (batch_size, channels, freqs, times)
        
        # Normalize features using min-max scaling
        normalized_features = normalize(features.numpy())
        print("Normalized feature shape: ", normalized_features.shape)
        
        # Standardize features using z-score calculation
        standardized_features = standardize(features.numpy())
        print("Standardized feature shape: ", standardized_features.shape)
        
        break  # Process only the first batch for demonstration

# Pipeline

In [ ]:
# raws = []

# participantsTSV = (
#     "/Users/nabijade/Desktop/Repositories/project/thessaloniki/participants.tsv"
# )

# # Read tab separated values (TSV) file into a DataFrame
# df = pd.read_csv(participantsTSV, sep="\t")

# # Convert Group labels to binary (A=1 for Alzheimer's, C=0 for controls/healthy)
# df["Group"].replace("A", 1, inplace=True)
# df["Group"].replace("C", 0, inplace=True)

# # Drop rows where Group is "F" (frontotemporal dementia)
# for x in df.index:
#     if df.loc[x, "Group"] == "F":
#         df.drop(x, inplace=True)

# # ====== EEG PIPELINE ======
# labels_list = []
# ids_list = []
# feature_list = []

# files = find_files("/Users/nabijade/Desktop/Repositories/project/thessaloniki", ".set")

# for i, file in enumerate(natsorted(files)):
#     if i > 63: # Avoid processing files with Frontotemporal Dementia (F) since they don't have labels in the dataframe
#         break
    
#     raw = mne.io.read_raw_eeglab(file)  
#     raw_clean = preprocess(raw)  
#     epochs = mne.make_fixed_length_epochs(raw_clean, duration=30, overlap=15)

#     freqs = mne.time_frequency.stftfreq(512, raw_clean.info['sfreq']) # freqs array of hz (0-512)
#     freq_mask = (freqs >= 0.5) & (freqs <= 45) # bool mask for freqs array (hz from 0.5-45 -> True)
    
#     # Use the patient ID and label from the dataframe
#     patient_id = df.iloc[i, 0]
#     patient_label = df.iloc[i, 3]
    
#     print(f"Processing patient_id: {patient_id}, label: {patient_label}")

#     for epoch in epochs:
#         # feature shape: (n_channels, n_freqs, n_times)
#         feature = abs(mne.time_frequency.stft(epoch, 512, 128)) 
        
#         # Filter the output from freq bins between 0.5-45 hz
#         feature = feature[:, freq_mask, :] # filter hz from 0.5 - 45
#         feature = pow(abs(feature), 2)
        
#         # Append the 3D feature to the list
#         feature_list.append(feature)
        
#         # Append the metadata once for EVERY epoch so the lengths match
#         labels_list.append(patient_label)
#         ids_list.append(patient_id)

# # Convert to numpy arrays
# labels_array = np.array(labels_list)
# ids_array = np.array(ids_list)
# feature_array = np.array(feature_list) # This will now be (Total_Epochs, Channels, Freqs, Times)

# # Save arrays
# np.save("labels_array.npy", labels_array)
# np.save("ids_array.npy", ids_array)
# np.save("feature_array.npy", feature_array)

# Dataset & DataLoader

In [6]:
# Custom Pytorch dataset
# 3 parameters: features, labels, indices
# __init__ method to initialize dataset with features, labels, and indices
class EEGDataset(Dataset):
    def __init__(self, features, labels, indices):
        self.features = torch.tensor(features)
        self.labels = torch.tensor(labels)
        self.indices = torch.tensor(indices)

    # __len__ len method to return length
    def __len__(self):
        return len(self.labels)

    # __ getitem__ method to return feature and label for a given index
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]
    
# Find unique patients
# train_test_split sklearn to split into: Train, Test, Val sets (80/10/10)
# Use np.isin and np.where to create boolean mask for each set and split arrays accordingly

unique_ids = np.unique(np.load("/Users/nabijade/Desktop/Repositories/project/processed/ids_array.npy"))

# # Split subject IDs array into 80/10/10 train/test/val sets
train_ids, test_val_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)
test_ids, val_ids = train_test_split(test_val_ids, test_size=0.5, random_state=42) 

train_bool_mask = np.isin(unique_ids, train_ids)
test_bool_mask = np.isin(unique_ids, test_ids)
val_bool_mask = np.isin(unique_ids, val_ids)

train_indices = np.where(train_bool_mask)[0]
test_indices = np.where(test_bool_mask)[0]
val_indices = np.where(val_bool_mask)[0]

# 3 dataset variables: train_dataset, test_dataset, val_dataset
# Parameters: features, labels, indices
train_dataset = EEGDataset(np.load("/Users/nabijade/Desktop/Repositories/project/processed/feature_array.npy"), np.load("/Users/nabijade/Desktop/Repositories/project/processed/labels_array.npy"), train_indices)
test_dataset = EEGDataset(np.load("/Users/nabijade/Desktop/Repositories/project/processed/feature_array.npy"), np.load("/Users/nabijade/Desktop/Repositories/project/processed/labels_array.npy"), test_indices)
val_dataset = EEGDataset(np.load("/Users/nabijade/Desktop/Repositories/project/processed/feature_array.npy"), np.load("/Users/nabijade/Desktop/Repositories/project/processed/labels_array.npy"), val_indices)

# 3 dataloader variables: train_dataloader, test_dataloader, val_dataloader
# Shuffle is true for only train, batchsize is hyperparameter to be tuned
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32)
val_dataloader = DataLoader(val_dataset, batch_size=32)

In [7]:
print("feature_array shape: ", np.load("/Users/nabijade/Desktop/Repositories/project/processed/feature_array.npy").shape)

feature_array shape:  (3435, 19, 23, 118)


# CNN Class
Source: 
https://discuss.pytorch.org/t/cnn-classification-for-4d-data-large-number-of-classes-training-accuracy-always-zero/54077


In [ ]:
class CSANet(nn.Module):

    def __init__(self):
        super().__init__()
        # 1st conv layer
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=128, kernel_size=5, padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.5)
        )
        
        # 2nd conv layer
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=5, padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.5)
        )
        

        # classifier
        self.fc = nn.Linear(128 * 107 * 28, 1)

    def forward(self, x):
        x = x.reshape(-1, 1, 19 * 23, 118)
        
        x = self.conv1(x)
        
        x = self.conv2(x)

        x = x.flatten(1)
        x = self.fc(x)
        x = torch.sigmoid(x)
        
        return x

In [ ]:
class ShuffleAttention(nn.Module):
    def __init__(self, channels=128, groups=8):
        super().__init__()
        self.groups = groups

        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        self.channel_fc = nn.Sequential(
            nn.Conv2d(channels // (2 * groups), channels // (2 * groups), kernel_size=1),
            nn.Sigmoid()
        )

        self.spatial_norm = nn.GroupNorm(
            num_groups=1,
            num_channels=channels // (2 * groups)
        )

        self.spatial_conv = nn.Sequential(
            nn.Conv2d(channels // (2 * groups), channels // (2 * groups), kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.reshape(-1)
        C, H, W = x.shape
        g = self.groups

        x = x.view(C // g, H, W)

        x_channel, x_spatial = torch.chunk(x, 2, dim=1)

        channel_weight = self.channel_fc(self.avg_pool(x_channel))
        x_channel = x_channel * channel_weight

        spatial_weight = self.spatial_conv(self.spatial_norm(x_spatial))
        x_spatial = x_spatial * spatial_weight

        out = torch.cat([x_channel, x_spatial], dim=1)
        out = out.view(C, H, W)
        out = self.ChannelShuffle(out)

        return out

# Training
Use Kaggle compute power

# Metrics